In [45]:
%matplotlib inline 
import random
import torch 
from d2l import torch as d2l 

In [22]:
def add(a,b):
    c = a+b
    return c

In [23]:
result = add(2,3)
result

5

In [32]:
def synthetic_data(w,b,num_example):#人工合成的数据，w权重（机器学习出来的） b基础值（偏置）样本的数量
    "生成 y=Xw + b + 噪声" #W代表了真实规律里的权重，机器学习自己找 w 和 b 
    X = torch.normal(0,1,(num_example,len(w))) #normal distribution 正态分布（高斯分布）
                                    #0 mean 平均值 这个随机数绕哪里波动，数字中心。#1 sigma 标准差 波动大小 控制随机数散开的程度
                                    #num_example 决定生成多少个数，以及形状是什么样的，样本特征有几个
    y = torch.matmul(X,w)+b         #线性回归模型的本身。输入x 算出预测结果y 
    #linreg模型                      #matmul = matrix multiplication 矩阵乘法
                                    #x 是样本的矩阵（两行两列），w是权重的向量（竖着排列 两列）矩阵*向量得到向量
    y += torch.normal(0,0.01,y.shape) #(平均值，标准差，shape)  生成一个形状和之前y向量结果一样的误差，以0为中心，标准差是0.01
    return X,y.reshape((-1,1))  #为什么这里的y需要reshpae因为X本身就是一个矩阵，y是一个向量（就是一维数组）但是机器学习里更喜欢二维矩阵

true_w = torch.tensor([2,-3.4])
true_b = 4.2 
features,labels =synthetic_data(true_w,true_b,1000)

In [42]:
print("features 形状:", features.shape)
print("labels 形状:", labels.shape)
print()
print("前5个样本的特征:")
print(features[:5])
print()
print("前5个样本的下山时间:")
print(labels[:5])

# 手算验证第0个样本
print()
print("手算验证第0个样本:")
print("特征:", features[0])
print("手算 (不含噪声):", features[0][0]*2 + features[0][1]*(-3.4) + 4.2)
print("实际 labels[0] (含噪声):", labels[0])
    

features 形状: torch.Size([1000, 2])
labels 形状: torch.Size([1000, 1])

前5个样本的特征:
tensor([[-1.6379,  0.2817],
        [ 0.3823, -0.6661],
        [-0.6952, -0.2859],
        [ 0.9387,  1.6834],
        [-0.0235, -0.2839]])

前5个样本的下山时间:
tensor([[-0.0169],
        [ 7.2318],
        [ 3.7694],
        [ 0.3493],
        [ 5.1082]])

手算验证第0个样本:
特征: tensor([-1.6379,  0.2817])
手算 (不含噪声): tensor(-0.0336)
实际 labels[0] (含噪声): tensor([-0.0169])


In [46]:
#定义一个data_iter函数，该函数接受批量大小，特征矩阵和标签向量作为输入，生成大小为batch_size的小批量
def data_iter(batch_size,features,labels):  
    num_examples = len(features)
    indices = list(range(num_examples))
    #这些样本是随机读取，没有特定的顺序
    random.shuffle(indices)
    for i in range(0, num_examples,batch_size):
        batch_indices = torch.tensor(
            indices[i:min(i+batch_size,num_examples)])        #这里的batch_indices是打乱清单上第0到第9位置上的值 batch还是索引，是在打乱清单上的索引的值
        yield features[batch_indices],labels[batch_indices]   #带yield的函数不叫函数叫生成器

batch_size = 10 
for x,y in data_iter(batch_size,features,labels):
            print(x,'\n',y)
            break
                        

tensor([[-0.2076,  0.3867],
        [-1.4935,  0.3403],
        [ 0.9111,  0.9887],
        [ 0.3569, -0.4727],
        [ 0.2289, -0.1478],
        [ 1.1474,  0.0966],
        [ 0.8149,  0.5048],
        [ 0.6603, -0.0271],
        [ 0.9290,  1.2487],
        [-0.0579, -0.1868]]) 
 tensor([[2.4631],
        [0.0660],
        [2.6451],
        [6.5218],
        [5.1643],
        [6.1640],
        [4.1326],
        [5.6190],
        [1.8020],
        [4.7043]])


In [47]:
print("indices 打乱前:", list(range(10)))

for x, y in data_iter(10, features, labels):
    print("x 形状:", x.shape)
    print("y 形状:", y.shape)
    break

indices 打乱前: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
x 形状: torch.Size([10, 2])
y 形状: torch.Size([10, 1])


In [49]:
print (x ,y )

tensor([[-0.4517,  0.7713],
        [ 0.2779,  0.4731],
        [-0.0493,  1.1408],
        [-1.0704, -0.0546],
        [ 1.2586,  0.2752],
        [-0.6670, -3.0772],
        [ 0.6318,  1.3779],
        [ 0.0201,  0.9758],
        [-0.9218,  1.6497],
        [-1.7355,  1.0667]]) tensor([[ 0.6583],
        [ 3.1591],
        [ 0.2268],
        [ 2.2503],
        [ 5.7893],
        [13.3400],
        [ 0.7633],
        [ 0.9171],
        [-3.2596],
        [-2.8826]])


In [50]:
print(x[0])
print(x[0])


tensor([-0.4517,  0.7713])
tensor([-0.4517,  0.7713])


In [51]:
for batch_num, (x, y) in enumerate(data_iter(10, features, labels)):
    print(f"第 {batch_num} 批, x形状 {x.shape}")
    if batch_num >= 2:
        break

第 0 批, x形状 torch.Size([10, 2])
第 1 批, x形状 torch.Size([10, 2])
第 2 批, x形状 torch.Size([10, 2])


In [53]:
# 方式1: 直接看整个数据集
print(features)          # jupyter会自动折叠中间部分, 只显示头尾
print(features.shape)    # (1000, 2)

# 方式2: 看指定范围
print(features[0:5])     # 前5个人
print(features[995:])    # 最后5个人
print(features[500])     # 第500个人

# 方式3: 强制打印全部1000行 (会刷屏, 慎用)
#torch.set_printoptions(threshold=10000)
#print(features)
#torch.set_printoptions(threshold=1000)   # 记得改回来

tensor([[-1.6379,  0.2817],
        [ 0.3823, -0.6661],
        [-0.6952, -0.2859],
        ...,
        [-0.3424, -0.5489],
        [ 0.1701, -0.2079],
        [-0.1228, -0.8716]])
torch.Size([1000, 2])
tensor([[-1.6379,  0.2817],
        [ 0.3823, -0.6661],
        [-0.6952, -0.2859],
        [ 0.9387,  1.6834],
        [-0.0235, -0.2839]])
tensor([[-0.4106,  0.9369],
        [-0.7349,  0.6130],
        [-0.3424, -0.5489],
        [ 0.1701, -0.2079],
        [-0.1228, -0.8716]])
tensor([-0.2753,  1.3606])


In [60]:
w = torch.normal(0,0.01,size=(2,1),requires_grad = True)
b = torch.zeros(1,requires_grad = True)       #这里的形状是b (1,)不是（1.1）这里就运用了之前学习的广播机制broadcasting

In [61]:
print("训练前的 w:", w.detach().flatten().tolist())
print("训练前的 b:", b.item())
print("真实值 true_w:", true_w.tolist(), " true_b:", true_b)

训练前的 w: [0.027868984267115593, -0.0022505209781229496]
训练前的 b: 0.0
真实值 true_w: [2.0, -3.4000000953674316]  true_b: 4.2


In [ ]:
#定义损失函数
def squared_loss(y_hat,y):   #loss到底算什么，为什么能告诉w怎么调整？ 模型负责猜答案，loss负责评价猜的有多离谱 loss告诉我们w,b往哪里调整，循环
    """均方损失。"""           #squared_loss :计算模型预测值和真实的标签之间的距离
    return (y_hat - y.reshape(y_hat.shape))**2/2  #这里就是预测值 y_hat - 真实值 y.reshape(y_hat.shape)
    #loss就是算的当前位置 